# Pipeline dữ liệu thời tiết và tín hiệu lũ Điện Biên

Notebook này chỉ điều phối pipeline đã được kiểm thử. Dữ liệu nghiệp vụ nằm trong Parquet/manifest; notebook không hard-code tọa độ hay logic cảnh báo.

In [ ]:
%pip install -q pandas pyarrow requests numpy

In [ ]:
import subprocess
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "data").is_dir() else cwd.parent
data_dir = repo_root / "data"
locations_path = data_dir / "dien_bien_locations.parquet"
if not locations_path.exists():
    raise FileNotFoundError(f"Không tìm thấy {locations_path}")

def run_data_script(script_name, *arguments):
    command = [sys.executable, str(data_dir / script_name), *map(str, arguments)]
    subprocess.run(command, cwd=repo_root, check=True)

# P0–P2: elevation → forecast snapshot → cảnh báo thời tiết MVP.
for script_name in [
    "download_elevation.py",
    "download_forecast.py",
    "alert_rules.py",
    "verify_weather_alert_mvp.py",
]:
    run_data_script(script_name)

# P3: node sông OSM → GloFAS daily → trend signal bổ trợ.
river_points_path = data_dir / "river_points.parquet"
if not river_points_path.exists():
    run_data_script("build_river_points.py")
for script_name in [
    "download_flood.py",
    "flood_signal.py",
    "verify_flood_signal.py",
]:
    run_data_script(script_name)

In [ ]:
overview_files = sorted(
    (data_dir / "alerts").glob(
        "snapshot_date=*/snapshot_time=*/new_admin_risk_overview.parquet"
    )
)
if not overview_files:
    raise FileNotFoundError("Chưa có new_admin_risk_overview.parquet")

risk_overview = pd.read_parquet(overview_files[-1])
flood_signals = pd.read_parquet(data_dir / "flood_signals.parquet")

display(
    risk_overview.sort_values(
        ["severity_rank", "new_admin_unit"],
        ascending=[False, True],
    )
)
display(
    flood_signals.loc[flood_signals["is_representative_grid_cell"]]
    .sort_values("peak_change_percent", ascending=False)
)